<a href="https://colab.research.google.com/github/Alina-cyber1/Detection_Number_Plate_KSA/blob/main/%D0%90%D0%B2%D1%82%D0%BE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import glob
from collections import Counter
from google.colab import drive

# 1. Подключаем Google Drive
print("Подключение к Google Drive...")
drive.mount('/content/drive')

# 2. Ищем папку number_plates
print("\nИщем папку с датасетом...")
search_path = '/content/drive/MyDrive'
dataset_path = None

for root, dirs, files in os.walk(search_path):
    if 'number_plates' in dirs:
        dataset_path = os.path.join(root, 'number_plates')
        break

if not dataset_path:
    print(" Папка 'number_plates' не найдена на Google Drive.")
else:
    print(f" Папка найдена: {dataset_path}")

    # 3. Запускаем аудит
    txt_files = glob.glob(os.path.join(dataset_path, '**', '*.txt'), recursive=True)
    class_counts = Counter()

    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_counts[int(parts[0])] += 1

    print(f"\nВсего файлов разметки: {len(txt_files)}")
    print(f"Уникальных ID классов: {len(class_counts)}")
    print(" Аудит пройден. Классов 27, можно переходить к подготовке.")

In [ ]:
import os
import shutil
import yaml
from sklearn.model_selection import train_test_split

# 1. СПИСОК КЛАССОВ (27 штук, ровно как в датасете)
CLASSES = ['0','1','2','3','4','5','6','7','8','9',
           'A','B','D','E','G','H','J','K','L','N',
           'R','S','T','U','V','X','Z']

# Пути к исходникам
SOURCE_PATH = dataset_path # Берем путь из Ячейки 1
TRAIN_DIR = os.path.join(SOURCE_PATH, 'train')
TEST_DIR = os.path.join(SOURCE_PATH, 'test')

# Создаем папку для нового датасета
NEW_DATASET = '/content/ksa_dataset'
if os.path.exists(NEW_DATASET):
    shutil.rmtree(NEW_DATASET) # Очищаем, если уже есть

os.makedirs(f'{NEW_DATASET}/train', exist_ok=True)
os.makedirs(f'{NEW_DATASET}/val', exist_ok=True)
os.makedirs(f'{NEW_DATASET}/test', exist_ok=True)

# 2. Разделяем train на train и val (80% / 20%)
print("Копируем файлы...")
all_images = [f for f in os.listdir(TRAIN_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
train_imgs, val_imgs = train_test_split(all_images, test_size=0.2, random_state=42)

def copy_data(file_list, src_dir, dst_dir):
    for img_name in file_list:
        base = os.path.splitext(img_name)[0]
        shutil.copy(os.path.join(src_dir, img_name), os.path.join(dst_dir, img_name))
        txt = base + '.txt'
        if os.path.exists(os.path.join(src_dir, txt)):
            shutil.copy(os.path.join(src_dir, txt), os.path.join(dst_dir, txt))

copy_data(train_imgs, TRAIN_DIR, f'{NEW_DATASET}/train')
copy_data(val_imgs, TRAIN_DIR, f'{NEW_DATASET}/val')

# Копируем тест
test_imgs = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
copy_data(test_imgs, TEST_DIR, f'{NEW_DATASET}/test')

# 3. Создаем правильный YAML
yaml_data = {
    'path': NEW_DATASET,
    'train': 'train',
    'val': 'val',
    'test': 'test',
    'names': {i: name for i, name in enumerate(CLASSES)}
}

with open(f'{NEW_DATASET}/data.yaml', 'w') as f:
    yaml.dump(yaml_data, f)

print(f" Готово: Train={len(train_imgs)}, Val={len(val_imgs)}, Test={len(test_imgs)}")

In [ ]:
!pip install ultralytics -q
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print(" Запуск обучения на 50 эпох...")

results = model.train(
    data=f'{NEW_DATASET}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    patience=15,
    device=0,
    project='KSA_Final',
    name='honest_run',
    fliplr=0.0,
    close_mosaic=10,
    exist_ok=True
)
print(" Обучение завершено!")

In [ ]:
import os
import cv2
import glob
import numpy as np
from ultralytics import YOLO

# Загружаем модель
BEST_MODEL_PATH = '/content/runs/detect/KSA_Final/honest_run/weights/best.pt'
best_model = YOLO(BEST_MODEL_PATH)

TEST_DIR = '/content/ksa_dataset/test'
test_imgs = sorted(glob.glob(os.path.join(TEST_DIR, '*.[jp]*g')))

correct_plates = 0
total_plates = 0
errors = []

print("СРАВНЕНИЕ ПРЕДСКАЗАНИЙ С ПРАВИЛЬНЫМИ ОТВЕТАМИ:\n")

for img_path in test_imgs:
    img_name = os.path.basename(img_path)
    base_name = os.path.splitext(img_name)[0]
    txt_path = os.path.join(TEST_DIR, base_name + '.txt')

    if not os.path.exists(txt_path):
        continue # Пропускаем, если нет разметки

    total_plates += 1

    # 1. Читаем ПРАВИЛЬНЫЙ ответ из .txt файла
    true_classes = []
    with open(txt_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                # В txt записан ID (0-26). Переводим его в букву через наш словарь
                class_id = int(parts[0])
                true_classes.append(best_model.names[class_id])
    true_text = "".join(true_classes)

    # 2. Получаем ПРЕДСКАЗАНИЕ от модели
    img = cv2.imread(img_path)
    h = img.shape[0]
    bottom = img[int(h * 0.5):h, :].copy()

    res = best_model(bottom, conf=0.25, verbose=False)[0]
    boxes = res.boxes.xyxy.cpu().numpy()
    classes = res.boxes.cls.cpu().numpy()

    pred_text = "Не найдено"
    if len(boxes) > 0:
        sorted_idx = boxes[:, 0].argsort()
        pred_chars = [best_model.names[int(c)] for c in classes[sorted_idx]]
        pred_text = "".join(pred_chars)

    # 3. Сравниваем
    if pred_text == true_text:
        correct_plates += 1
        print(f" {img_name}: {pred_text} == {true_text}")
    else:
        errors.append((img_name, true_text, pred_text))
        print(f" {img_name}: Предсказано '{pred_text}', а должно быть '{true_text}'")

print("\n" + "="*50)
print(f"ИТОГОВАЯ ТОЧНОСТЬ (Full Plate Accuracy):")
print(f"Правильно прочитано: {correct_plates} из {total_plates} фото")
print(f"Точность: {(correct_plates / total_plates) * 100:.1f}%")
print("="*50)

In [ ]:
import os
import glob

DATASET_PATH = '/content/ksa_dataset'

def clean_dataset(split_name):
    split_path = os.path.join(DATASET_PATH, split_name)
    txt_files = glob.glob(os.path.join(split_path, '*.txt'))

    removed_count = 0
    for txt_file in txt_files:
        with open(txt_file, 'r') as f:
            lines = f.readlines()

        # Считаем количество размеченных объектов
        if len(lines) < 4 or len(lines) > 8:
            # Удаляем и txt, и картинку
            os.remove(txt_file)
            # Ищем картинку с таким же именем (любое расширение)
            base = txt_file.rsplit('.', 1)[0]
            for ext in ['.jpg', '.jpeg', '.png']:
                img_path = base + ext
                if os.path.exists(img_path):
                    os.remove(img_path)
                    break
            removed_count += 1

    print(f" {split_name}: удалено {removed_count} мусорных файлов")

clean_dataset('train')
clean_dataset('val')

# Пересоздаем кэш (чтобы YOLO не подхватил удаленные файлы)
for split in ['train', 'val']:
    cache_file = os.path.join(DATASET_PATH, f'{split}.cache')
    if os.path.exists(cache_file):
        os.remove(cache_file)
print(" Кэш очищен")

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
print(" Запуск обучения (Production Level)...")

results = model.train(
    data=f'{DATASET_PATH}/data.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    patience=15,
    device=0,
    project='KSA_Production_Final',
    name='clean_run',
    fliplr=0.0,          # Запрет на зеркалирование текста
    close_mosaic=10,     # Мозаика отключается в конце
    hsv_v=0.4,           # Аугментация яркости (для темных фото)
    hsv_s=0.7,           # Аугментация насыщенности
    erasing=0.2,         # Затирание (для устойчивости к бликам)
    exist_ok=True
)
print(" Обучение завершено!")

In [ ]:
import os
import cv2
import glob
import random
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

BEST_MODEL_PATH = '/content/runs/detect/KSA_Production_Final/clean_run/weights/best.pt'
best_model = YOLO(BEST_MODEL_PATH)

def recognize_plate_final(img_path, conf=0.25):
    img = cv2.imread(img_path)
    if img is None: return None, "Ошибка чтения", []

    h, w = img.shape[:2]
    bottom = img[int(h * 0.5):h, :].copy()

    res = best_model(bottom, conf=conf, verbose=False)[0]
    boxes = res.boxes.xyxy.cpu().numpy()
    classes = res.boxes.cls.cpu().numpy()
    confs = res.boxes.conf.cpu().numpy()

    if len(boxes) == 0:
        return bottom, "Ничего не найдено", []

    sorted_idx = boxes[:, 0].argsort()
    boxes = boxes[sorted_idx]
    chars = [best_model.names[int(c)] for c in classes[sorted_idx]]
    confs = confs[sorted_idx]

    # Фильтрация полосок
    valid_boxes, valid_chars, valid_confs = [], [], []
    for box, char, c in zip(boxes, chars, confs):
        if (box[2] - box[0]) / (box[3] - box[1]) > 0.15:
            valid_boxes.append(box)
            valid_chars.append(char)
            valid_confs.append(c)

    if not valid_boxes:
        return bottom, "Ничего не найдено", []

    # Формат 4 цифры + 3 буквы
    digits = []
    letters = []
    for box, char, c in zip(valid_boxes, valid_chars, valid_confs):
        if char.isdigit() and len(digits) < 4:
            digits.append((char, c))
        elif char.isalpha() and len(letters) < 3 and len(digits) > 0:
            letters.append((char, c))

    final_sequence = digits + letters
    text = "".join([c for c, _ in final_sequence])

    # --- ФОРМИРОВАНИЕ ГИПОТЕЗ ---
    hypotheses = []
    if len(text) < 7:
        return bottom, f"Отказ: Найдено {len(text)} из 7 символов", [text]

    # Проверяем уверенность
    for char, c in final_sequence:
        if c < 0.5:
            hypotheses.append(f"Символ '{char}' размыт (уверенность {c:.2f}). Возможно, это другая буква.")

    if hypotheses:
        verdict = f"ОТКАЗ. Гипотеза: {text}. Причина: размытые символы."
    else:
        verdict = text

    # --- ОТРИСОВКА (Аккуратная) ---
    result_img = bottom.copy()
    for box in valid_boxes:
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(result_img, (x1, y1), (x2, y2), (0, 255, 0), 2)

    # Пишем текст МАЛЕНЬКИМ шрифтом В САМОМ ВЕРХУ (чтобы не закрывать номер)
    cv2.putText(result_img, verdict, (5, 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1)

    return result_img, verdict, hypotheses

# Тестируем
TEST_DIR = '/content/ksa_dataset/test'
all_imgs = sorted(glob.glob(os.path.join(TEST_DIR, '*.[jp]*g')))
random.seed(42)
random_imgs = random.sample(all_imgs, 5)

print("ФИНАЛЬНЫЕ РЕЗУЛЬТАТЫ С ГИПОТЕЗАМИ:\n")
for i, img_path in enumerate(random_imgs):
    result_img, verdict, hyps = recognize_plate_final(img_path)

    print(f"[{i+1}] {os.path.basename(img_path)}")
    print(f"    Вердикт: {verdict}")
    for h in hyps:
        print(f"    - {h}")
    print("-" * 50)

    plt.figure(figsize=(10, 4))
    plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.show()